In [1]:
import sys
sys.path.append("../")

##### import library

In [2]:
from typing import List, Optional
import requests
import pandas as pd
import mygene
import utility
import importlib
importlib.reload(utility)
from utility import resolve_ensembl_ids_with_fallback, get_chembl_ids_fast,  get_disease_ids_fast, validate_id_dataframe
import pickle

In [3]:
dct_result = dict()

##### Read pickle

In [4]:
with open("Biochatter_same_question_response_final.pkl", "rb") as f:
    Biochatter_same_question_response = pickle.load(f)


In [5]:
for model in ['llama-3.3-70b-versatile', 'gpt-5-nano', 'grok-4-1-fast-non-reasoning-latest', "gemini-2.5-flash-lite", "OpenTarget", "BioChatter", "ctd", "ttd", "hcdt"]:

    
    # if model=="llama-3.3-70b-versatile":
    #     dct_result[model] = Llama_same_question_response[model]

    # if model=="gpt-5-nano":
    #     dct_result[model] = OpenAI_same_question_response[model]

    # if model=='grok-4-1-fast-non-reasoning-latest':
    #     dct_result[model] = Grok_same_question_response[model]

    # if model=="gemini-2.5-flash-lite":
    #     dct_result[model] = Gemini_same_question_response[model]

    # if model=="OpenTarget":
    #     dct_result[model] = Opentarget_same_question_response[model]

    if model=="BioChatter":
        dct_result[model] = Biochatter_same_question_response[model]
    # if model=="ctd":
    #     dct_result[model] = CTD_same_question_response[model]

    # if model=="hcdt":
    #     dct_result[model] = HCDT_same_question_response[model]


##### Verify if only valid columns are present

In [6]:
allowed = {"gene_name", "drug_name", "disease_name", "pathway_name"}

missing_df = []
not_dataframe = []
empty_df = []
bad_columns = []

total_payloads = 0

for model, queries in dct_result.items():
    if not isinstance(queries, dict):
        continue

    for question, runs in queries.items():
        if not isinstance(runs, dict):
            continue

        for run_id, payload in runs.items():
            total_payloads += 1

            if not isinstance(payload, dict) or "dataframe" not in payload:
                missing_df.append({
                    "model": model,
                    "question": question,
                    "run_id": run_id,
                    "payload_type": type(payload).__name__
                })
                continue

            df = payload.get("dataframe")

            if not isinstance(df, pd.DataFrame):
                not_dataframe.append({
                    "model": model,
                    "question": question,
                    "run_id": run_id,
                    "df_type": type(df).__name__
                })
                continue

            if df.empty:
                empty_df.append({
                    "model": model,
                    "question": question,
                    "run_id": run_id,
                    "columns": list(df.columns)
                })

            cols = {str(c) for c in df.columns}
            extra = sorted(cols - allowed)

            if extra:
                bad_columns.append({
                    "model": model,
                    "question": question,
                    "run_id": run_id,
                    "columns": list(df.columns),
                    "extra_columns": extra
                })

print(f"Total payloads checked: {total_payloads}")
print(f"Missing dataframe key: {len(missing_df)}")
print(f"Dataframe is not pd.DataFrame: {len(not_dataframe)}")
print(f"Empty dataframes: {len(empty_df)}")
print(f"Dataframes with disallowed columns: {len(bad_columns)}")

bad_columns_df = pd.DataFrame(bad_columns)
missing_df_df = pd.DataFrame(missing_df)
not_dataframe_df = pd.DataFrame(not_dataframe)
empty_df_df = pd.DataFrame(empty_df)

display(bad_columns_df)
display(missing_df_df)
display(not_dataframe_df)
display(empty_df_df)


Total payloads checked: 342
Missing dataframe key: 0
Dataframe is not pd.DataFrame: 0
Empty dataframes: 0
Dataframes with disallowed columns: 0


""


""


""


""


In [7]:
dct_jaccard = dict()

question_entities = {
    "gene_name": set(),
    "drug_name": set(),
    "disease_name": set(),
}

allowed_cols = {"gene_name", "drug_name", "disease_name"}

for model, queries in dct_result.items():
    dct_jaccard[model] = {}
    print(f"\nWorking for model: {model}")

    for question, runs in queries.items():

        for run_id, payload in runs.items():
            df = payload.get("dataframe")

            # ---- validation ----
            if not isinstance(df, pd.DataFrame) or df.empty:
                continue

            # ---- skip pathway outputs ----
            if "pathway_name" in df.columns:
                continue

            # ---- column sanity ----
            if not set(df.columns).issubset(allowed_cols):
                continue

            # ---- collect entities globally ----
            for col in allowed_cols:
                if col in df.columns:
                    values = (
                        df[col]
                        .dropna()
                        .astype(str)
                        .str.strip()
                        .tolist()
                    )
                    question_entities[col].update(values)

# ---- finalize (sets → sorted lists) ----
for k in question_entities:
    question_entities[k] = sorted(question_entities[k])


Working for model: BioChatter


##### Associate id with all gene entry

In [8]:
# Resolve unique gene symbols to IDs (MyGene -> OpenTargets fallback).
df_gene = resolve_ensembl_ids_with_fallback(
    list(question_entities["gene_name"]),
    use_opentargets_fallback=True,
)
df_gene = df_gene.drop_duplicates(subset=["gene_name", "gene_id"]).reset_index(drop=True)
df_gene


2026-02-27 17:40:15,853 [WARNING] Input sequence provided is already in string format. No operation performed
2026-02-27 17:40:15,854 [WARNING] Input sequence provided is already in string format. No operation performed
2026-02-27 17:40:15,855 [INFO] querying 1-261 ...
2026-02-27 17:40:17,594 [INFO] HTTP Request: POST https://mygene.info/v3/query/ "HTTP/1.1 200 OK"
2026-02-27 17:40:18,598 [INFO] Finished.
2026-02-27 17:40:18,605 [WARNING] 2 input query terms found no hit:	['ERBB1', 'G72']


,gene_name,gene_id,source
0,ABCA7,ENSG00000064687,mygene
1,ABL1,ENSG00000097007,mygene
2,ABL2,ENSG00000143322,mygene
3,ADNP,ENSG00000101126,mygene
4,AKT1,ENSG00000142208,mygene
...,...,...,...
256,VCP,ENSG00000165280,mygene
257,VDR,ENSG00000111424,mygene
258,VHL,ENSG00000134086,mygene
259,VRK2,ENSG00000028116,mygene


In [9]:
df_gene[df_gene.isna().any(axis=1)]

,gene_name,gene_id,source


##### Associate id with all drug entry

In [10]:
# Resolve unique drug surface forms to IDs.
# This keeps original raw names as rows and writes mapped ID + source.
df_drug = await get_chembl_ids_fast(list(question_entities["drug_name"]))
df_drug = df_drug.drop_duplicates(subset=["drug_name", "drug_id"]).reset_index(drop=True)
df_drug


2026-02-27 17:40:34,809 [INFO] [LLM] Total 12 names split into 1 batches (max_batch_chars=6000, max_batch_items=80, model=llama-3.3-70b-versatile).
2026-02-27 17:40:34,810 [INFO] [LLM] Batch size: 12


[map][drug][OpenTargets] raw='5-fluorouracil' -> id='CHEMBL185'
[map][drug][OpenTargets] raw='Abatacept' -> id='CHEMBL1201823'
[map][drug][OpenTargets] raw='Adalimumab' -> id='CHEMBL1201580'
[map][drug][OpenTargets] raw='Afatinib' -> id='CHEMBL1173655'
[map][drug][OpenTargets] raw='Amantadine' -> id='CHEMBL660'
[map][drug][OpenTargets] raw='Amivantamab' -> id='CHEMBL4297774'
[map][drug][OpenTargets] raw='Anakinra' -> id='CHEMBL1201570'
[map][drug][OpenTargets] raw='Apomorphine' -> id='CHEMBL53'
[map][drug][OpenTargets] raw='Azathioprine' -> id='CHEMBL1542'
[map][drug][OpenTargets] raw='Baricitinib' -> id='CHEMBL2105759'
[map][drug][OpenTargets] raw='Belimumab' -> id='CHEMBL1789843'
[map][drug][OpenTargets] raw='Benztropine' -> id='CHEMBL1201203'
[map][drug][OpenTargets] raw='Bromocriptine' -> id='CHEMBL493'
[map][drug][OpenTargets] raw='Budesonide' -> id='CHEMBL1370'
[map][drug][OpenTargets] raw='Calcitriol' -> id='CHEMBL846'
[map][drug][OpenTargets] raw='Carbidopa' -> id='CHEMBL120074

2026-02-27 17:40:35,554 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-27 17:40:35,577 [INFO] [LLM] Batch done in 0.77s
2026-02-27 17:40:35,578 [INFO] [OT] Verifying 2 unique drug suggestions...
2026-02-27 17:40:35,990 [INFO] [OT] 'mercaptopurine' → 'MERCAPTOPURINE' (CHEMBL1200751)
2026-02-27 17:40:35,999 [INFO] Done: 1 / 12 confirmed | 11 → None


[map][drug][Llama] raw='6-mercaptopurine' -> canonical='MERCAPTOPURINE' -> id='CHEMBL1200751'


,drug_name,drug_id,source
0,5-fluorouracil,CHEMBL185,OpenTargets
1,6-mercaptopurine,CHEMBL1200751,Llama
2,Abatacept,CHEMBL1201823,OpenTargets
3,Adalimumab,CHEMBL1201580,OpenTargets
4,Afatinib,CHEMBL1173655,OpenTargets
...,...,...,...
236,valdecoxib,CHEMBL865,OpenTargets
237,vandetanib,CHEMBL24828,OpenTargets
238,vedolizumab,CHEMBL1743087,OpenTargets
239,voclosporin,CHEMBL5314379,OpenTargets


In [11]:
df_drug["source"].value_counts()

source
OpenTargets    229
Llama            1
Name: count, dtype: int64

In [12]:
# df_drug[df_drug["source"]=="Llama"]

In [13]:
df_drug[df_drug.isna().any(axis=1)]

,drug_name,drug_id,source
17,Carbidopa/levodopa,None,None
70,adalimumb,None,None
149,lassmiditan,None,None
210,sorafenib BRAF,None,None
211,sorafenib FLT3,None,None
212,sorafenib FLT4,None,None
213,sorafenib KDR,None,None
214,sorafenib KIT,None,None
215,sorafenib PDGFRB,None,None
216,sorafenib RAF1,None,None


##### Associate id with all disease entry

In [14]:
# Resolve unique disease surface forms to IDs.
# Canonical fallback is internal; returned rows remain keyed by raw input disease_name.
df_disease = await get_disease_ids_fast(list(question_entities["disease_name"]))
df_disease = df_disease.drop_duplicates(subset=["disease_name", "disease_id"]).reset_index(drop=True)
df_disease


2026-02-27 17:40:37,255 [INFO] [LLM] Total 15 names split into 1 batches (max_batch_chars=6000, max_batch_items=80, model=llama-3.3-70b-versatile).
2026-02-27 17:40:37,256 [INFO] [LLM] Batch size: 15


[map][disease][OpenTargets] raw='Acute myeloid leukemia' -> id='EFO_0000222'
[map][disease][OpenTargets] raw='Acute myeloid leukemia with mutated NPM1' -> id='MONDO_0044923'
[map][disease][OpenTargets] raw='Adrenocortical carcinoma' -> id='EFO_1000796'
[map][disease][OpenTargets] raw='Alzheimer's disease' -> id='MONDO_0007088'
[map][disease][OpenTargets] raw='Amyotrophic lateral sclerosis' -> id='MONDO_0004976'
[map][disease][OpenTargets] raw='Anaplastic large cell lymphoma' -> id='EFO_0003032'
[map][disease][OpenTargets] raw='Anaplastic thyroid carcinoma' -> id='EFO_1000595'
[map][disease][OpenTargets] raw='Ankylosing spondylitis' -> id='EFO_0003898'
[map][disease][OpenTargets] raw='Apert syndrome' -> id='MONDO_0007041'
[map][disease][OpenTargets] raw='Autism spectrum disorder' -> id='EFO_0003756'
[map][disease][OpenTargets] raw='Beare-Stevenson cutis gyrata syndrome' -> id='MONDO_0007412'
[map][disease][OpenTargets] raw='Beare-Stevenson syndrome' -> id='MONDO_0007412'
[map][disease][

2026-02-27 17:40:38,028 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-27 17:40:38,032 [INFO] [LLM] Batch done in 0.77s
2026-02-27 17:40:38,032 [INFO] [LLM] 'Cardiofacio-cutaneous syndrome' → 'cardiofaciocutaneous syndrome'
2026-02-27 17:40:38,033 [INFO] [LLM] 'HER2-positive breast cancer' → 'breast cancer'
2026-02-27 17:40:38,033 [INFO] [LLM] 'HER2-positive gastric cancer' → 'gastric cancer'
2026-02-27 17:40:38,034 [INFO] [LLM] 'HER2-positive gastric or gastroesophageal junction adenocarcinoma' → 'gastric cancer'
2026-02-27 17:40:38,034 [INFO] [LLM] 'HER2-positive gastric/GEJ adenocarcinoma' → 'gastric cancer'
2026-02-27 17:40:38,034 [INFO] [LLM] 'HER2-positive gastroesophageal junction cancer' → 'gastroesophageal junction cancer'
2026-02-27 17:40:38,035 [INFO] [LLM] 'HER2-positive metastatic breast cancer' → 'breast cancer'
2026-02-27 17:40:38,035 [INFO] [LLM] 'Low-grade serous ovarian carcinoma' → 'ovarian cancer'
2026-02-27 17:40

[map][disease][Llama] raw='Cardiofacio-cutaneous syndrome' -> canonical='cardiofaciocutaneous syndrome' -> id='MONDO_0015280'
[map][disease][Llama] raw='HER2-positive breast cancer' -> canonical='breast carcinoma' -> id='EFO_0000305'
[map][disease][Llama] raw='HER2-positive gastric cancer' -> canonical='stomach neoplasm' -> id='EFO_0003897'
[map][disease][Llama] raw='HER2-positive gastric or gastroesophageal junction adenocarcinoma' -> canonical='stomach neoplasm' -> id='EFO_0003897'
[map][disease][Llama] raw='HER2-positive gastric/GEJ adenocarcinoma' -> canonical='stomach neoplasm' -> id='EFO_0003897'
[map][disease][Llama] raw='HER2-positive metastatic breast cancer' -> canonical='breast carcinoma' -> id='EFO_0000305'
[map][disease][Llama] raw='Low-grade serous ovarian carcinoma' -> canonical='ovarian carcinoma' -> id='EFO_0001075'
[map][disease][Llama] raw='Metastatic gastric cancer' -> canonical='stomach neoplasm' -> id='EFO_0003897'
[map][disease][Llama] raw='Non-metastatic castrat

,disease_name,disease_id,source
0,Acute myeloid leukemia,EFO_0000222,OpenTargets
1,Acute myeloid leukemia with mutated NPM1,MONDO_0044923,OpenTargets
2,Adrenocortical carcinoma,EFO_1000796,OpenTargets
3,Alzheimer's disease,MONDO_0007088,OpenTargets
4,Amyotrophic lateral sclerosis,MONDO_0004976,OpenTargets
...,...,...,...
146,systemic lupus erythematosus,MONDO_0007915,OpenTargets
147,tuberculosis,MONDO_0018076,OpenTargets
148,ulcerative colitis,EFO_0000729,OpenTargets
149,urothelial carcinoma,EFO_0008528,OpenTargets


In [15]:
df_disease["source"].value_counts()

source
OpenTargets    136
Llama           10
Name: count, dtype: int64

In [16]:
df_disease[df_disease.isna().any(axis=1)]

,disease_name,disease_id,source
44,HER2-positive gastroesophageal junction cancer,None,None
66,MSI-H/dMMR solid tumors,None,None
71,Metastatic gastroesophageal junction adenocarc...,None,None
79,Noninfectious uveitis,None,None
139,postoperative pain,None,None


In [17]:
# ------------------------------------------------------------------
# Disease canonical audit (OpenTargets-strict IDs only)
# ------------------------------------------------------------------
# IMPORTANT:
# - This cell runs an additional Llama canonicalization pass over unresolved rows.
# - IDs are still resolved ONLY via OpenTargets mapIds (score==1).
# - To avoid changing the baseline resolver output, writeback is OFF by default.
APPLY_AUDIT_MAPPINGS = False

# 1) Rows unresolved after primary resolver
unresolved_rows = df_disease[df_disease["disease_id"].isna()].reset_index(drop=True)
unresolved_raw_names = list(
    dict.fromkeys(
        str(name).strip()
        for name in unresolved_rows["disease_name"].dropna().astype(str).tolist()
        if str(name).strip()
    )
)

# 2) Llama canonicalization suggestions
disease_input_map = {name: None for name in unresolved_raw_names}
canonicalisation_map = await utility.canonicalise_disease_dict(disease_input_map)

# 3) OpenTargets strict verification (score==1 only)
canonical_terms_norm = list(
    dict.fromkeys(
        utility._normalize_disease_term(v)
        for v in canonicalisation_map.values()
        if isinstance(v, str) and v.strip()
    )
)
canonical_to_id = utility.resolve_diseases_opentargets_bulk(
    canonical_terms_norm,
    strict_score_one=True,
)

# 4) Build persistent audit table
audit_rows = []
applied_count = 0
candidate_count = 0

for raw_disease_name in unresolved_raw_names:
    canonical_name = canonicalisation_map.get(raw_disease_name)
    if isinstance(canonical_name, str) and canonical_name.strip():
        canonical_name = canonical_name.strip()
        canonical_norm = utility._normalize_disease_term(canonical_name)
        resolved_id = canonical_to_id.get(canonical_norm)
    else:
        canonical_name = None
        canonical_norm = None
        resolved_id = None

    can_apply = bool(resolved_id)
    if can_apply:
        candidate_count += 1
        print(f"[audit][disease][Llama->OT] raw='{raw_disease_name}' -> canonical='{canonical_name}' -> id='{resolved_id}'")

    if can_apply and APPLY_AUDIT_MAPPINGS:
        mask = df_disease["disease_name"].astype(str).str.strip() == raw_disease_name
        df_disease.loc[mask, "disease_id"] = resolved_id
        df_disease.loc[mask, "source"] = "Llama"
        applied_count += 1

    status = (
        "mappable_ot_exact" if can_apply else
        ("no_canonical" if canonical_name is None else "canonical_not_in_ot")
    )
    audit_rows.append({
        "raw_disease_name": raw_disease_name,
        "canonical_name": canonical_name,
        "canonical_norm": canonical_norm,
        "resolved_disease_id": resolved_id,
        "status": status,
        "can_apply": can_apply,
        "applied": bool(can_apply and APPLY_AUDIT_MAPPINGS),
    })

disease_canonical_audit_df = pd.DataFrame(audit_rows)
if not disease_canonical_audit_df.empty:
    disease_canonical_audit_df = disease_canonical_audit_df.sort_values(
        ["status", "raw_disease_name"],
        ascending=[True, True],
    ).reset_index(drop=True)

print(f"[summary][disease][audit] OT-mappable candidates: {candidate_count}")
print(f"[summary][disease][audit] applied to df_disease: {applied_count} (APPLY_AUDIT_MAPPINGS={APPLY_AUDIT_MAPPINGS})")
if not disease_canonical_audit_df.empty:
    print("[summary][disease][audit] status counts:")
    print(disease_canonical_audit_df["status"].value_counts(dropna=False))

disease_canonical_audit_df


2026-02-27 17:40:38,959 [INFO] [LLM] Total 5 names split into 1 batches (max_batch_chars=6000, max_batch_items=80, model=llama-3.3-70b-versatile).
2026-02-27 17:40:38,960 [INFO] [LLM] Batch size: 5
2026-02-27 17:40:39,284 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-27 17:40:39,286 [INFO] [LLM] Batch done in 0.33s
2026-02-27 17:40:39,287 [INFO] [LLM] 'HER2-positive gastroesophageal junction cancer' → None
2026-02-27 17:40:39,287 [INFO] [LLM] 'MSI-H/dMMR solid tumors' → None
2026-02-27 17:40:39,288 [INFO] [LLM] 'Metastatic gastroesophageal junction adenocarcinoma' → None
2026-02-27 17:40:39,288 [INFO] [LLM] 'Noninfectious uveitis' → 'uveitis'
2026-02-27 17:40:39,288 [INFO] [LLM] 'postoperative pain' → None
2026-02-27 17:40:39,289 [INFO] [OT] Verifying 1 unique suggestions...
2026-02-27 17:40:39,684 [INFO] [OT] 'uveitis' → 'uveitis' (EFO_1001231)
2026-02-27 17:40:39,686 [INFO] Done: 1 / 5 confirmed | 4 → None


[audit][disease][Llama->OT] raw='Noninfectious uveitis' -> canonical='uveitis' -> id='EFO_1001231'
[summary][disease][audit] OT-mappable candidates: 1
[summary][disease][audit] applied to df_disease: 0 (APPLY_AUDIT_MAPPINGS=False)
[summary][disease][audit] status counts:
status
no_canonical         4
mappable_ot_exact    1
Name: count, dtype: int64


,raw_disease_name,canonical_name,canonical_norm,resolved_disease_id,status,can_apply,applied
0,Noninfectious uveitis,uveitis,uveitis,EFO_1001231,mappable_ot_exact,True,False
1,HER2-positive gastroesophageal junction cancer,None,None,None,no_canonical,False,False
2,MSI-H/dMMR solid tumors,None,None,None,no_canonical,False,False
3,Metastatic gastroesophageal junction adenocarc...,None,None,None,no_canonical,False,False
4,postoperative pain,None,None,None,no_canonical,False,False


In [18]:
# disease_canonical_audit_df

In [19]:
df_disease[df_disease["source"]=='Llama']

,disease_name,disease_id,source
14,Cardiofacio-cutaneous syndrome,MONDO_0015280,Llama
40,HER2-positive breast cancer,EFO_0000305,Llama
41,HER2-positive gastric cancer,EFO_0003897,Llama
42,HER2-positive gastric or gastroesophageal junc...,EFO_0003897,Llama
43,HER2-positive gastric/GEJ adenocarcinoma,EFO_0003897,Llama
45,HER2-positive metastatic breast cancer,EFO_0000305,Llama
63,Low-grade serous ovarian carcinoma,EFO_0001075,Llama
70,Metastatic gastric cancer,EFO_0003897,Llama
76,Non-metastatic castration-resistant prostate c...,EFO_0001663,Llama
128,fallelopian tube cancer,EFO_1000251,Llama


In [20]:
df_disease["source"].value_counts()

source
OpenTargets    136
Llama           10
Name: count, dtype: int64

In [21]:
df_disease

,disease_name,disease_id,source
0,Acute myeloid leukemia,EFO_0000222,OpenTargets
1,Acute myeloid leukemia with mutated NPM1,MONDO_0044923,OpenTargets
2,Adrenocortical carcinoma,EFO_1000796,OpenTargets
3,Alzheimer's disease,MONDO_0007088,OpenTargets
4,Amyotrophic lateral sclerosis,MONDO_0004976,OpenTargets
...,...,...,...
146,systemic lupus erythematosus,MONDO_0007915,OpenTargets
147,tuberculosis,MONDO_0018076,OpenTargets
148,ulcerative colitis,EFO_0000729,OpenTargets
149,urothelial carcinoma,EFO_0008528,OpenTargets


In [22]:
# df_disease[df_disease["source"]=="Llama"]

In [23]:
def compute_jaccard_from_run_dataframes(dct_result, df_gene, df_disease, df_drug):
    """
    Compute per-question cross-run Jaccard consistency after mapping entities to IDs.

    For each (model, question):
    - intersection: entries present in ALL valid runs
    - union: entries present in ANY valid run
    - jaccard = |intersection| / |union|

    All comparisons are CASE-INDEPENDENT and ID-based.

    Side effects:
    - Stores per-run mapped dataframe in:
        dct_result[model][question][run]['dataframe_id']

    Returns:
    - dct_jaccard[model][question] = {
          'jaccard': float,
          'n_valid_runs': int,
          'intersection': set[tuple],
          'union': set[tuple],
      }
    """

    dct_jaccard = {}

    # ------------------------------------------------------------------
    # Build deterministic one-to-one mapping tables
    # ------------------------------------------------------------------
    gene_map = (
        df_gene.assign(_gene_norm=df_gene["gene_name"].astype(str).str.strip().str.upper())
        [["_gene_norm", "gene_id"]]
        .dropna(subset=["gene_id"])
        .drop_duplicates("_gene_norm", keep="first")
    )

    disease_map = (
        df_disease.assign(_disease_norm=df_disease["disease_name"].astype(str).str.strip().str.upper())
        [["_disease_norm", "disease_id"]]
        .dropna(subset=["disease_id"])
        .drop_duplicates("_disease_norm", keep="first")
    )

    drug_map = (
        df_drug.assign(_drug_norm=df_drug["drug_name"].astype(str).str.strip().str.upper())
        [["_drug_norm", "drug_id"]]
        .dropna(subset=["drug_id"])
        .drop_duplicates("_drug_norm", keep="first")
    )

    # ------------------------------------------------------------------
    # Main loop
    # ------------------------------------------------------------------
    for model, model_payload in dct_result.items():
        dct_jaccard[model] = {}
        print(f"\nWorking for model: {model}")

        for question_key, runs_dict in model_payload.items():
            run_sets = {}

            for run_number, run_payload in runs_dict.items():
                df = run_payload.get("dataframe")

                # ---- basic validity ----
                if not isinstance(df, pd.DataFrame) or df.empty:
                    continue

                if "pathway_name" in df.columns:
                    continue

                work_df = df.copy()

                # print(work_df.head())

                final_cols = []

                # ---- gene ----
                if "gene_name" in work_df.columns:
                    work_df["_gene_norm"] = (
                        work_df["gene_name"].astype(str).str.strip().str.upper()
                    )
                    work_df = work_df.merge(gene_map, on="_gene_norm", how="left")
                    final_cols.append("gene_id")

                # ---- disease ----
                if "disease_name" in work_df.columns:
                    work_df["_disease_norm"] = (
                        work_df["disease_name"].astype(str).str.strip().str.upper()
                    )
                    work_df = work_df.merge(disease_map, on="_disease_norm", how="left")
                    final_cols.append("disease_id")

                # ---- drug ----
                if "drug_name" in work_df.columns:
                    work_df["_drug_norm"] = (
                        work_df["drug_name"].astype(str).str.strip().str.upper()
                    )
                    work_df = work_df.merge(drug_map, on="_drug_norm", how="left")
                    final_cols.append("drug_id")

                if not final_cols:
                    continue

                # ---- canonical ID dataframe ----
                id_df = (
                    work_df[final_cols]
                    .dropna(how="any")
                    .drop_duplicates()
                    .reset_index(drop=True)
                )

                dct_result[model][question_key][run_number]["dataframe_id"] = id_df

                if id_df.empty:
                    continue

                # ---- CASE-INDEPENDENT tuple set ----
                run_set = {
                    tuple(str(v).strip().upper() for v in row)
                    for row in id_df.itertuples(index=False, name=None)
                }

                if run_set:
                    run_sets[run_number] = run_set

            # ------------------------------------------------------------------
            # Compute intersection / union
            # ------------------------------------------------------------------
            n_valid_runs = len(run_sets)

            if n_valid_runs < 2:
                print(
                    f"Not enough valid runs for '{question_key}' "
                    f"({n_valid_runs}/{len(runs_dict)}). Skipping."
                )
                continue

            sets = list(run_sets.values())
            intersection = set.intersection(*sets)
            union = set.union(*sets)

            if not union:
                print(f"Empty union for '{question_key}', skipping.")
                continue

            jaccard = len(intersection) / len(union)

            dct_jaccard[model][question_key] = {
                "jaccard": jaccard,
                "n_valid_runs": n_valid_runs,
                "intersection": intersection,
                "union": union,
            }

            print(
                f"'{question_key}': "
                f"J={jaccard:.4f}, "
                f"|∩|={len(intersection)}, "
                f"|∪|={len(union)}, "
                f"runs={n_valid_runs}"
            )

    return dct_jaccard


dct_jaccard = compute_jaccard_from_run_dataframes(
    dct_result=dct_result,
    df_gene=df_gene,
    df_disease=df_disease,
    df_drug=df_drug,
)


Working for model: BioChatter
'Which diseases are associated with BRAF?': J=0.4286, |∩|=6, |∪|=14, runs=5
'Which genes are associated with amyotrophic lateral sclerosis?': J=0.5000, |∩|=11, |∪|=22, runs=5
'Which diseases are associated with CDK4?': J=0.2000, |∩|=1, |∪|=5, runs=5
Not enough valid runs for 'Which pathways are associated with the JAK2 gene?' (0/5). Skipping.
'Which drugs are used to treat rheumatoid arthritis?': J=0.9412, |∩|=16, |∪|=17, runs=5
'Which genes are associated with ovarian cancer?': J=0.7143, |∩|=10, |∪|=14, runs=5
'What are the known targets of regorafenib?': J=0.8462, |∩|=11, |∪|=13, runs=5
Not enough valid runs for 'Which pathways are associated with the STAT3 gene?' (0/5). Skipping.
'Which genes are associated with fever (pyrexia)?': J=0.0000, |∩|=0, |∪|=15, runs=5
'Which diseases are treated with bevacizumab?': J=0.5000, |∩|=5, |∪|=10, runs=5
'What are the known targets of cabozantinib?': J=1.0000, |∩|=5, |∪|=5, runs=5
Not enough valid runs for 'Which pa

In [24]:
# Run Jaccard consistency computation across runs after ID grounding.
dct_jaccard = compute_jaccard_from_run_dataframes(
    dct_result=dct_result,
    df_gene=df_gene,
    df_disease=df_disease,
    df_drug=df_drug,
)

# Flatten into a quick summary table for inspection.
rows = [
    {
        "model": model,
        "question": question,
        "jaccard": payload["jaccard"],
        "n_valid_runs": payload["n_valid_runs"],
        "intersection_size": len(payload["intersection"]),
        "union_size": len(payload["union"]),
    }
    for model, qmap in dct_jaccard.items()
    for question, payload in qmap.items()
]

jaccard_summary = pd.DataFrame(rows)
if jaccard_summary.empty:
    print("No valid Jaccard entries were produced. Check input runs and validation filters.")
else:
    display(jaccard_summary.sort_values(["model", "jaccard"], ascending=[True, False]).head(20))



Working for model: BioChatter
'Which diseases are associated with BRAF?': J=0.4286, |∩|=6, |∪|=14, runs=5
'Which genes are associated with amyotrophic lateral sclerosis?': J=0.5000, |∩|=11, |∪|=22, runs=5
'Which diseases are associated with CDK4?': J=0.2000, |∩|=1, |∪|=5, runs=5
Not enough valid runs for 'Which pathways are associated with the JAK2 gene?' (0/5). Skipping.
'Which drugs are used to treat rheumatoid arthritis?': J=0.9412, |∩|=16, |∪|=17, runs=5
'Which genes are associated with ovarian cancer?': J=0.7143, |∩|=10, |∪|=14, runs=5
'What are the known targets of regorafenib?': J=0.8462, |∩|=11, |∪|=13, runs=5
Not enough valid runs for 'Which pathways are associated with the STAT3 gene?' (0/5). Skipping.
'Which genes are associated with fever (pyrexia)?': J=0.0000, |∩|=0, |∪|=15, runs=5
'Which diseases are treated with bevacizumab?': J=0.5000, |∩|=5, |∪|=10, runs=5
'What are the known targets of cabozantinib?': J=1.0000, |∩|=5, |∪|=5, runs=5
Not enough valid runs for 'Which pa

,model,question,jaccard,n_valid_runs,intersection_size,union_size
8,BioChatter,What are the known targets of cabozantinib?,1.000000,5,5,5
14,BioChatter,For which diseases is darolutamide used?,1.000000,5,1,1
26,BioChatter,Which biological targets does sorafenib intera...,1.000000,4,8,8
31,BioChatter,Which diseases are associated with HTT?,1.000000,5,1,1
34,BioChatter,Which genes are associated with cystic fibrosis?,1.000000,5,1,1
36,BioChatter,Which diseases are treated with metformin?,1.000000,5,2,2
39,BioChatter,Which diseases are treated with sorafenib?,1.000000,5,3,3
43,BioChatter,What are the known targets of abatacept?,1.000000,5,2,2
53,BioChatter,Which genes are associated with hepatitis C?,1.000000,5,2,2
57,BioChatter,Which drugs target CTLA4?,1.000000,5,2,2


In [25]:
# Persist enriched run payloads with dataframe_id attached for each run.
with open("Biochatter_same_question_response_with_id.pkl", "wb") as f:
    pickle.dump(dct_result, f)
